### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](../resources/monkey.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
# Alternative manual download link: https://yadi.sk/d/_nGyU2IajjR9-w
!wget "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O arxivData.json.tar.gz
!tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

--2026-10-05 16:41:58--  https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.67.18, 2620:100:6023:18::a27d:4312
Connecting to www.dropbox.com (www.dropbox.com)|162.125.67.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1 [following]
--2026-10-05 16:41:58--  https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc16acbdaff7fab505c8f03a0a9d.dl.dropboxusercontent.com/cd/0/inline/DJa5XC7unNrkZRYlT1t5CbHZSrN8i2aEsYRHCvMN3FyjMNZaBfgpy3NJw_QS-xxuM38urMoKXpZHXC8n_AOMZZ-Fl8xCGN7h04HosjGufVhvboejGw3PNDhaVbh_TAsUAYE/file?dl=1# [following]
--2026-10-05 16:41:59--  https://uc16acbdaff7fab505c8f03a0a9d.dl.dropbox

,author,day,id,link,month,summary,tag,title,year
7960,"[{'name': 'Jongpil Lee'}, {'name': 'Jiyoung Pa...",6,1703.01789v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,"Recently, the end-to-end approach that learns ...","[{'term': 'cs.SD', 'scheme': 'http://arxiv.org...",Sample-level Deep Convolutional Neural Network...,2017
4209,"[{'name': 'Kihyuk Sohn'}, {'name': 'Sifei Liu'...",7,1708.02191v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",8,"Despite rapid advances in face recognition, th...","[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Unsupervised Domain Adaptation for Face Recogn...,2017
27679,"[{'name': 'Mohammad Reza Khosravi'}, {'name': ...",8,1704.02455v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",4,Remote sensing image processing is so importan...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",A New Pseudo-color Technique Based on Intensit...,2017
1574,"[{'name': 'Zachary Lipton'}, {'name': 'Xiujun ...",15,1711.05715v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",11,We present a new algorithm that significantly ...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",BBQ-Networks: Efficient Exploration in Deep Re...,2017
33590,"[{'name': 'Mohammad Soleimani-Pouri'}, {'name'...",28,1311.7213v1,"[{'rel': 'related', 'href': 'http://dx.doi.org...",11,Interaction between users in online social net...,"[{'term': 'cs.SI', 'scheme': 'http://arxiv.org...",Finding a Maximum Clique using Ant Colony Opti...,2013


In [3]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [4]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer
from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()

lines = [' '.join(tokenizer.tokenize(line.lower())) for line in lines]

In [5]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:

$$
    P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1})
$$ 

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$
    P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})
$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words. 

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [7]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens:
# - `UNK` represents absent tokens,
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - every prefix must contain exactly (n - 1) tokens. If a prefix is shorter than that,
      pad it with UNK on the left. The number of padding tokens therefore depends on n:
      derive it from n, do not hard-code it. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
      For n=2 the very same prefixes give (UNK,), (the,) and (approach,), and for n=1 all
      of them give the empty tuple (). Equivalently: prepend (n - 1) UNK tokens to a line,
      then slide a window of size n over it.
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.

    Useful invariant: a line with L tokens always contributes exactly L + 1 counted events
    (its L tokens plus EOS), for every n. Note also that NGramLanguageModel below already
    implements this padding rule in get_possible_next_tokens - keep the two consistent, or
    the model will look up prefixes that you never counted.
    """
    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] = how many times word3 occured after (word1, word2)

    for line in lines:
        tokens = [UNK] * (n-1) + line.split() + [EOS]
        for i in range(n-1, len(tokens)):
            counts[tuple(tokens[i-n+1:i])][tokens[i]] += 1          

    return counts


In [8]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

# the padding rule must hold for every n, not just for n=3
for n in (1, 2, 3, 5, 10):
    counts_n = count_ngrams(dummy_lines, n=n)
    assert set(map(len, counts_n.keys())) == {n - 1}, \
        "every prefix must have exactly (n - 1) tokens, got lengths %s for n=%i" % (
            set(map(len, counts_n.keys())), n)
    assert sum(sum(c.values()) for c in counts_n.values()) == 2877, \
        "each line must contribute exactly len(tokens) + 1 events (its tokens plus EOS) " \
        "for every n. If this passes for n=3 but fails otherwise, you have most likely " \
        "hard-coded the number of UNK padding tokens instead of deriving it from n."

Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [9]:
class NGramLanguageModel:
    def __init__(self, lines, n):
        """
        Train a simple count-based language model:
        compute probabilities P(w_t | prefix) given ngram counts

        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n

        counts = count_ngrams(lines, self.n)

        # compute token proabilities given counts
        self.probs = defaultdict(Counter)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)

        # populate self.probs with actual probabilities
        for prefix in counts:
            words = counts[prefix]

            self.probs[prefix] = {word : count / sum(words.values()) 
                                 for word, count in words.items()}

    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]

    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

Let's test it!

In [10]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [11]:
lm = NGramLanguageModel(lines, n=3)

The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [12]:
def get_next_token(lm, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """

    possible_next_tokens = lm.get_possible_next_tokens(prefix)
    tokens, probs = zip(*possible_next_tokens.items())

    tokens = np.array(tokens)
    probs = np.array(probs, dtype=np.float64)
    
    if temperature == 0:
        return tokens[np.argmax(probs)]
    
    probs = probs ** (1.0 / temperature)
    probs = probs / np.sum(probs)
    
    return np.random.choice(tokens, p=probs)

In [13]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Looks nice!


Let's have fun with this model

In [14]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

artificial neural network ( rnn ) based on compression algorithms are to be np - complete , without correction , we present a self - taught first and 1 - e simulation are to have large variation between frames as possible from the captured image and video clips like vines present a method to mitigate the effectiveness of non - linear physics , data points collect gradually and some rely on powerful gpu . due to their population counterparts , even though there are only defined for nodes in a worked out , to real scenes . expectations are challenging to


In [15]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

bridging the gap between the two methods for the presence of latent variables in the context of the proposed method . _EOS_


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $-\frac1N$, where $N$ is the __total number of scored tokens__ in the corpora.

A sentence $w_1 \dots w_L$ contributes $L + 1$ factors to that product, because EOS is predicted and scored exactly like a regular token:

$$P(w_1 \mid \varnothing) \cdot P(w_2 \mid w_1) \cdots P(w_L \mid w_1 \dots w_{L-1}) \cdot P(EOS \mid w_1 \dots w_L)$$

Here $\varnothing$ is the empty prefix. EOS is counted in $N$ as well, so $N = \sum_{\text{sentences}} (L + 1)$ -- the same number of events you counted in `count_ngrams`.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [16]:
def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above

    Note: scoring a line "w_1 w_2 ... w_L" means scoring L + 1 events:
        P(w_1 | ""), P(w_2 | "w_1"), ..., P(w_L | "w_1 ... w_{L-1}"), P(EOS | "w_1 ... w_L")
    The prefix of the very first token is the empty string "", i.e. an empty prefix and not
    a string of UNKs: the model pads it to (n - 1) UNK tokens on its own inside
    get_possible_next_tokens, so do not add any padding here. Likewise, w_1 is simply the
    first real token of the line. The trailing EOS is scored like any other token and is
    counted in N too, so a line with L tokens adds L + 1 to N.

    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    total_log_prob = 0
    total_tokens = 0
    for line in lines:
        tokens = line.split()

        for i, token in enumerate(tokens):
            prefix = ' '.join(tokens[:i])
            proba = lm.get_next_token_prob(prefix, token)
            
            if proba <= 0:
                log_proba = min_logprob
            else:
                log_proba = max(min_logprob, np.log(proba))

            total_log_prob += log_proba

        unk_proba = lm.get_next_token_prob(' '.join(tokens), EOS)
        if proba <= 0:
            log_proba = min_logprob
        else:
            log_proba = max(min_logprob, np.log(unk_proba))
        total_log_prob += log_proba

        total_tokens += len(tokens) + 1

    return np.exp(-total_log_prob / total_tokens)

In [26]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [25]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

In [ ]:
for n in (1, 2):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

In [ ]:
# whoops, it just blew up :)

### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [17]:
class LaplaceLanguageModel(NGramLanguageModel):
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))


**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [31]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [16]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

N = 1, Perplexity = 1835.56092
N = 2, Perplexity = 470.48021
N = 3, Perplexity = 3679.44765


In [17]:
prefix = 'Happy'

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

Happy revising mend posters atal mgss staleness carver choosing help ratsabychesterieee2012 oncina esai preiss rank1elimkl manner observers unfound 1303 typings dinger alendronate understating 312 vsc modrl unlinked zomorodian guidefill niching capacitating facewarehouse exit quasiconvexity ogis busy cii sand golem discounts reversal emosim508 ocean lbs drafting clarifai metacomputing autostereogram suspends globalization automation conterexamples 6803 lenging theyare lexicalised written landed $[ railstation pthreads fpn scoped nlogspace reaffirm deci nuage rivaled contained alfa ids envisat looming transation omniglot supervisée mimetic srelu decoys texturation iors daisy dcg indoors poverty dogmatism softmax repot enthusiasts lbie recognizers gprfs provers _i 1549 lumen protege emit decomposes whittaker fishes


### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $\lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1. Discounting takes $\delta$ away from each of the $N_{1+}(prefix_{n-1}, \bullet)$ __distinct__ tokens ever seen after that prefix, and that is exactly the mass to redistribute:
$$\lambda_{prefix_{n-1}} = {\delta \cdot N_{1+}(prefix_{n-1}, \bullet) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)}$$
- $P_{kn}(w_t | prefix_{n-2})$ is the lower-order model: Kneser-Ney smoothing for the {n-1}-gram language model, i.e. the same formula with the oldest token dropped from the prefix.
- Unigram $P_{kn}(w_t)$ is a special case that terminates the recursion: how likely it is to see $w_t$ in an unfamiliar context

__Prefixes that were never seen in training.__ If $Count(prefix_{n-1}, \hat w) = 0$ for every $\hat w$, then the denominator is zero and both the first term and $\lambda_{prefix_{n-1}}$ come out as $\tfrac{0}{0}$. There is simply no higher-order evidence to interpolate with, so back off completely: take $\lambda_{prefix_{n-1}} = 1$ and the first term equal to $0$, which means returning $P_{kn}(w_t | prefix_{n-2})$ unchanged. The result is still a valid probability distribution, and this is the case you will hit constantly at test time.

__Choosing $\delta$.__ Keep $\delta \in (0, 1]$. Counts are integers $\ge 1$, so within that range $\max(0, Count - \delta) = Count - \delta$ and the probabilities sum to exactly 1. With $\delta > 1$ the $\max$ clips away less mass than $\lambda$ hands back to the lower order, and the distribution stops normalizing -- the validity test below will fail.

See lecture slides or wiki for more detailed formulae.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

_Note: the test below iterates over `dummy_lm.vocab`, so your class has to expose a `self.vocab` set holding every known token, just like `LaplaceLanguageModel` does._

In [19]:
class KneserNeyLanguageModel(NGramLanguageModel):
    """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        # hint: you will need n-gram counts for every order from 1 to n, plus a self.vocab set

        self.n = n
        self.delta = delta
        self.counts = {i: count_ngrams(lines, i) for i in range(1, n+1)}
        unigram_counts = self.counts[1][()]
        self.vocab = set(unigram_counts.keys())
        self.vocab.add(EOS)
        self.vocab.add(UNK)

        self.totals = {}
        self.lambdas = {}

        for k in range(2, n+1):
            for prefix, word_counts in self.counts[k].items():
                total = sum(word_counts.values())
                n1plus = len(word_counts)

                self.totals[(k, prefix)] = total
                self.lambdas[(k, prefix)] = self.delta * n1plus / total

        # вероятности униграмм для выхода из рекурсии
        unigram_total = sum(unigram_counts.values())
        self.unigram_probs = {word: count / unigram_total for word, count in unigram_counts.items()}

    def _prepare(self, prefix):
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix

        return tuple(prefix)

    def _prob(self, prefix, token):
        if len(prefix) == 0: return self.unigram_probs.get(token, 0)

        k = len(prefix) + 1
        if ((k, prefix) not in self.totals): return self._prob(prefix[1:], token)
        return (max(0, self.counts[k][prefix][token] - self.delta) / self.totals[(k, prefix)] + 
                self.lambdas[(k, prefix)] * self._prob(prefix[1:], token))
    
    def get_possible_next_tokens(self, prefix):
        prefix = self._prepare(prefix)

        return {word: self._prob(prefix, word) for word in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        # remember to handle a prefix that never occurred in training, see above

        prefix = self._prepare(prefix)
        return self._prob(prefix, next_token)

In [23]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [26]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=0.7)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

N = 1, Perplexity = 1835.12284
N = 2, Perplexity = 403.36375
N = 3, Perplexity = 309.30316
